In [1]:
# Initialize Otter
import otter
grader = otter.Notebook("DS3000-9000_A04.ipynb")

# Assignment 4: Classification

**DS 3000A/9000A Fall 2026**

This assignment will be graded on Gradescope.

1. Run `pip install otter-grader` first, if you don't have it installed.
2. Included in this notebook are public test cases, which you can use to verify your answers before submitting.
3. Additional private test cases will run on Gradescope upon submission. You will see your score but not the tests themselves.
4. You can revise and re-upload your submission on Gradescope up until the deadline.

In [ ]:
## Install packages if neccessary.
## Uncomment and run if you need to.

# !pip install numpy scipy pandas matplotlib

In [2]:
import numpy as np
import scipy.stats as st

---

## Question 1
 
**50 Points**
 
In the lab, evaluation metrics were computed using `scikit-learn`. Here, you will implement them from scratch.
 
Given the confusion matrix entries (True Positives (TP), True Negatives (TN), False Positives (FP), and False Negatives (FN)) the metrics are:
 
$$
\text{Accuracy} = \frac{TP + TN}{N}, \qquad
\text{Precision} = \frac{TP}{TP + FP}, \qquad
\text{Recall} = \frac{TP}{TP + FN} ,
$$

where $N$ is the number of data points.
 
$$
\text{Specificity} = \frac{TN}{TN + FP}, \qquad
\text{F1} = 2 \cdot \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}
$$
 
We use the following synthetic imbalanced dataset.
`y_true` is drawn from a heavily imbalanced distribution (95% negative, 5% positive). `y_pred_model` simulates a naive random classifier. 
`y_pred_degenerate` always predicts the majority class. 
Your goal is to show that high accuracy does not imply a useful classifier.

In [3]:
np.random.seed(30009000)
N = 1000
y_true = np.random.choice([0, 1], size=N, p=[0.95, 0.05])
y_pred_model = (np.random.rand(N) < 0.6).astype(int)
y_pred_degenerate = np.zeros(N, dtype=int)

1. Implement `confusion_matrix_counts(y_true, y_pred)` that returns `(TP, TN, FP, FN)` as Python `int`s.

2. Implement `accuracy(y_true, y_pred)`, `precision(y_true, y_pred)`, `recall(y_true, y_pred)`, `specificity(y_true, y_pred)`, and `f1(y_true, y_pred)` as Python `float`s. Return `0.0` when the denominator is zero.

3. Assign:
   - `acc_model`, `rec_model`, `f1_model` for `y_pred_model`.
   - `acc_deg`, `rec_deg`, `f1_deg` for `y_pred_degenerate`.

In [21]:
def confusion_matrix_counts(
    y_true: np.ndarray, y_pred: np.ndarray
) -> tuple[int, int, int, int]:
    """Compute TP, TN, FP, FN from true and predicted labels.

    Args:
        y_true: True binary labels, shape (N,).
        y_pred: Predicted binary labels, shape (N,).

    Returns:
        Tuple (TP, TN, FP, FN) as Python ints.
    """
    match = y_true[y_true == y_pred]
    noMatch = y_true[y_true != y_pred]
    TP = match[match == 1]
    TN = match[match == 0]
    FP = noMatch[noMatch == 0]
    FN = noMatch[noMatch == 1]
    
    return len(TP), len(TN), len(FP), len(FN)

def accuracy(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Accuracy score.

    Args:
        y_true: True binary labels, shape (N,).
        y_pred: Predicted binary labels, shape (N,).

    Returns:
        Accuracy as a float.
    """
    TP, TN, FP, FN = confusion_matrix_counts(y_true, y_pred)
    return float((TP + TN) / (TP + TN + FP + FN))

def precision(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Precision score.

    Args:
        y_true: True binary labels, shape (N,).
        y_pred: Predicted binary labels, shape (N,).

    Returns:
        Precision as a float. Returns 0.0 if TP + FP == 0.
    """
    TP, TN, FP, FN = confusion_matrix_counts(y_true, y_pred)
    return float(TP / (TP + FP)) if TP + FP != 0 else 0.0

def recall(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Recall score.

    Args:
        y_true: True binary labels, shape (N,).
        y_pred: Predicted binary labels, shape (N,).

    Returns:
        Recall as a float. Returns 0.0 if TP + FN == 0.
    """
    TP, TN, FP, FN = confusion_matrix_counts(y_true, y_pred)
    return TP / (TP + FN) if TP + FN != 0 else 0.0

def specificity(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Specificity score.

    Args:
        y_true: True binary labels, shape (N,).
        y_pred: Predicted binary labels, shape (N,).

    Returns:
        Specificity as a float. Returns 0.0 if TN + FP == 0.
    """
    TP, TN, FP, FN = confusion_matrix_counts(y_true, y_pred)
    return TN/(TN + FP) if TN + FP != 0 else 0.0

def f1(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """F1 score.

    Args:
        y_true: True binary labels, shape (N,).
        y_pred: Predicted binary labels, shape (N,).

    Returns:
        F1 score as a float. Returns 0.0 if precision + recall == 0.
    """
    r = recall(y_true, y_pred)
    p = precision(y_true, y_pred)
    return 2 * (p * r) / (p + r)  if p + r != 0 else 0.0


acc_model = accuracy(y_true, y_pred_model)
rec_model = recall(y_true, y_pred_model)
f1_model  = f1(y_true, y_pred_model)
acc_deg   = accuracy(y_true, y_pred_degenerate)
rec_deg   = recall(y_true, y_pred_degenerate)
f1_deg    = f1(y_true, y_pred_degenerate)

x = confusion_matrix_counts(y_true, y_pred_degenerate)
print(len(y_true), len(y_pred_degenerate), sum(x), x[0], x[1], x[2], x[3])
print(
    f"Model    --- Accuracy: {acc_model:.3f}, Recall: {rec_model:.3f}, F1: {f1_model:.3f}"
)
print(f"Degenerate -- Accuracy: {acc_deg:.3f}, Recall: {rec_deg:.3f}, F1: {f1_deg:.3f}")

1000 1000 1000 0 953 0 47
Model    --- Accuracy: 0.436, Recall: 0.617, F1: 0.093
Degenerate -- Accuracy: 0.953, Recall: 0.000, F1: 0.000


In [22]:
grader.check("q1")

q1 results: All test cases passed!

---

## Question 2
 
**50 Points**
 
For multi-class classification with $K$ classes, logistic regression generalizes via the **softmax** function $\mathbb{R}^K \to \Delta^{K-1}$:
 
$$
\text{softmax}(\mathbf{z})_k = \frac{\exp(z_k)}{\sum_{j=0}^{K-1} \exp(z_j)}, \quad k = 0, \ldots, K-1
$$
 
The model is $f(\mathbf{x}; \mathbf{W}) = \mathbf{W}\phi(\mathbf{x})$ where $\mathbf{W} \in \mathbb{R}^{K \times D}$, and the log-likelihood is:
 
$$
\log p(\mathcal{D} \mid \mathbf{W}) = \sum_{n=1}^N \sum_{k=0}^{K-1} \mathbb{1}[y_n = k] \log \text{softmax}(f(\mathbf{x}_n; \mathbf{W}))_k
$$
 
We use the Iris dataset:

In [23]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

np.random.seed(30009000)

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

K = 3  # Iris has 3 classes
D = X_train.shape[1]

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

(120, 4) (120,)
(30, 4) (30,)


1. Implement `softmax(z)` for a single input vector $\mathbf{z} \in \mathbb{R}^K$.
   - Numerical stability trick: subtract `z.max()` from `z` before exponentiating.

2. Implement `log_softmax(z)` **without** calling `softmax(z)`. You should derive the log of the softmax function by hand first.

3. Implement `predict_proba(X, W)` that returns the $N \times K$ matrix of class probabilities, where each row is `softmax(W @ x_n)`. Use the bias trick: append a column of ones to `X` inside the function.

4. Implement `log_likelihood_categorical(W, X, y)`. Use `log_softmax` inside, not `log(predict_proba(...))`.

In [ ]:
def softmax(z: np.ndarray) -> np.ndarray:
    """Softmax function for a single vector.

    Args:
        z: Input vector of shape (K,).

    Returns:
        Probability vector of shape (K,), sums to 1.
    """
    total = np.sum(np.exp(z))
    res = np.zeros_like(z)
    for i in range(z):
        res[i] =  np.exp(z[i]) / total
    return res     
        

def log_softmax(z: np.ndarray) -> np.ndarray:
    """Log-softmax function for a single vector.

    Do NOT call softmax(z) inside this function.

    Args:
        z: Input vector of shape (K,).

    Returns:
        Log-probability vector of shape (K,).
    """
    ...


def predict_proba(X: np.ndarray, W: np.ndarray) -> np.ndarray:
    """Predict class probabilities.

    Args:
        X: Input matrix of shape (N, D). Bias NOT yet appended.
        W: Weight matrix of shape (K, D+1).

    Returns:
        Probability matrix of shape (N, K).
    """
    ...


def log_likelihood_categorical(W: np.ndarray, X: np.ndarray, y: np.ndarray) -> float:
    """Categorical log-likelihood for a softmax regression model.

    Args:
        W: Weight matrix of shape (K, D+1).
        X: Input matrix of shape (N, D).
        y: Integer class labels of shape (N,).

    Returns:
        Log-likelihood as a float.
    """
    ...

In [ ]:
grader.check("q2")

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

In [ ]:
# Save your notebook first, then run this cell to export your submission.
grader.export(pdf=False, run_tests=True)